<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**RULE:**
"Prioritize content that is both stale (not updated in >90 days) and underperforming in CTR relative to its position. If content has high visibility (impressions) but users aren't clicking despite a good rank, it’s a 'Quick Win' for a refresh."

**Reason Codes:**
CTR_UNDERPERFORM: CTR is below 0.5% despite being in a top-20 position.
STALE_CONTENT: Content hasn't been updated in over 90 days.
HIGH_STALE_DECOY: High impressions but hasn't been touched in 6 months.

**Why I chose these thresholds:**


- I used 90 days because content that has not been updated for more than three months is more likely to become outdated.
- I used a CTR threshold of 0.5% because pages ranking in the top 20 should generally receive more clicks. A lower CTR suggests that the title, meta description, or content may need improvement.
- I also required at least 500 impressions so that very low-traffic pages do not trigger the rule based on noisy data.

In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df)}, base decline rate: {base_rate:.3f}")

rows: 30000, base decline rate: 0.542


In [3]:
# Signal 1: staleness (behind FlyRank's refresh flags) ---
order = ["0-30", "31-90", "91-180", "181+"]
staleness_table = (df.groupby("freshness_tier")
                      .agg(n=("is_declining_label", "size"),
                           decline_rate=("is_declining_label", "mean"))
                      .reindex(order))
print(staleness_table)

                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


VERDICT: CONFIRMED

The decline rate increases from newer freshness tiers to older ones. The 181+ day bucket has a noticeably higher decline rate than the 0–30 day bucket, which supports using content freshness as one of the scoring signals.

In [4]:
# Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
df["low_ctr_flag"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

ctr_table = (df.groupby("low_ctr_flag")
               .agg(n=("is_declining_label", "size"),
                    decline_rate=("is_declining_label", "mean")))
print(ctr_table)

                  n  decline_rate
low_ctr_flag                     
False         20241      0.501062
True           9759      0.627113


VERDICT: CONFIRMED

Pages with low CTR while ranking in the top 20 have a higher decline rate than pages without this flag. This supports using CTR performance as another signal in the baseline rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# Encoding the rules
df["stale_flag"] = df["freshness_tier"].isin(["91-180", "181+"])
df["score"] = df["stale_flag"].astype(int) + df["low_ctr_flag"].astype(int)

def get_reason(row):
    if row["stale_flag"] and row["low_ctr_flag"]: return "STALE_AND_LOW_CTR"
    if row["low_ctr_flag"]: return "CTR_UNDERPERFORM"
    if row["stale_flag"]: return "STALE_CONTENT"
    return "NO_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)
df["action"] = df["score"].apply(lambda s: "REFRESH_NOW" if s == 2 else ("REVIEW" if s == 1 else "NO_ACTION"))

# Sorting and writing CSV
queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "score", "reason_code", "action", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Top Precision@50: {queue.head(50)['is_declining_label'].mean():.2%}")

Top Precision@50: 36.00%


In [7]:
!head work/outputs/baseline_action_score.csv

content_id,client_id,rank,score,reason_code,action,is_declining_label
content_5fe46e04994d,client_4e07408562,1,2,STALE_AND_LOW_CTR,REFRESH_NOW,1
content_cb112fce36be,client_19581e27de,2,2,STALE_AND_LOW_CTR,REFRESH_NOW,1
content_36ff89c8214e,client_19581e27de,3,2,STALE_AND_LOW_CTR,REFRESH_NOW,0
content_c21024970297,client_19581e27de,4,2,STALE_AND_LOW_CTR,REFRESH_NOW,0
content_c8e9d6ab9013,client_19581e27de,5,2,STALE_AND_LOW_CTR,REFRESH_NOW,1
content_d17681677e69,client_19581e27de,6,2,STALE_AND_LOW_CTR,REFRESH_NOW,0
content_a7427266c305,client_19581e27de,7,2,STALE_AND_LOW_CTR,REFRESH_NOW,0
content_c5063073d048,client_6208ef0f77,8,2,STALE_AND_LOW_CTR,REFRESH_NOW,0
content_3d94572c3a35,client_19581e27de,9,2,STALE_AND_LOW_CTR,REFRESH_NOW,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Code to display top 20 for review
review_cols = ["content_id", "rank", "score", "reason_code", "action",
               "freshness_tier", "avg_position", "ctr", "impressions_90d"]
print(queue[review_cols].head(10).to_string(index=False))

          content_id  rank  score       reason_code      action freshness_tier  avg_position  ctr  impressions_90d
content_5fe46e04994d     1      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           4.2 0.14           517715
content_cb112fce36be     2      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           5.6 0.16           309910
content_36ff89c8214e     3      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           7.3 0.05           295097
content_c21024970297     4      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           5.1 0.41           211366
content_c8e9d6ab9013     5      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           9.7 0.00           208678
content_d17681677e69     6      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           5.8 0.24           201584
content_a7427266c305     7      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180           5.7 0.11           201111
content_c5063073d048     8      2 STALE_AND_LOW_CTR REFRESH_NOW         91-180  

| Rank | Action      | Reason                                                                       | Confidence | What would make it wrong?                                                                           |
| ---- | ----------- | ---------------------------------------------------------------------------- | ---------- | --------------------------------------------------------------------------------------------------- |
| 1    | REFRESH_NOW | Page is stale (181+) and CTR is below 0.5% while receiving many impressions. | High       | Users may already get the answer from Google's featured snippet, so refreshing may not improve CTR. |
| 2    | REFRESH_NOW | Stale content with low CTR and strong visibility.                            | High       | The page may intentionally target informational queries where lower CTR is normal.                  |
| 3    | REFRESH_NOW | Both refresh signals are triggered.                                          | High       | Only the title/meta description may need improvement rather than the content.                       |
| 4    | REFRESH_NOW | High impressions, low CTR, stale content.                                    | High       | Competitors may have richer search results that reduce clicks.                                      |
| 5    | REFRESH_NOW | Content is old and underperforming.                                          | High       | The topic may be losing search demand rather than needing a refresh.                                |
| 6    | REVIEW      | Only CTR signal triggered.                                                   | Medium     | The page is still new and Google rankings may still be stabilizing.                                 |
| 7    | REVIEW      | CTR below threshold.                                                         | Medium     | Low CTR could be caused by branded competitors instead of poor content.                             |
| 8    | REVIEW      | CTR slightly below threshold.                                                | Medium     | Tracking or reporting errors could affect CTR values.                                               |
| 9    | REVIEW      | Stale content only.                                                          | Medium     | The page may be evergreen and still satisfying user intent.                                         |
| 10   | REVIEW      | Stale but not strongly underperforming.                                      | Low        | Refreshing could waste effort if traffic is already stable.                                         |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
print("Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr")
print("trend_direction / trend_pct used only to build is_declining_label (the label) —",
      "never inside stale_flag, low_ctr_flag, score, reason_code, or action.")

borderline = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline[review_cols])

Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr
trend_direction / trend_pct used only to build is_declining_label (the label) — never inside stale_flag, low_ctr_flag, score, reason_code, or action.
                 content_id   rank  score    reason_code  action  \
15470  content_36e7b91747fa  15471      1  STALE_CONTENT  REVIEW   
15471  content_3b50b0c27334  15472      1  STALE_CONTENT  REVIEW   
15482  content_75bbe9161b5d  15483      1  STALE_CONTENT  REVIEW   

                                confidence  days_since_last_update  \
15470  Medium — meets one signal condition                     211   
15471  Medium — meets one signal condition                     104   
15482  Medium — meets one signal condition                     102   

       avg_position  ctr  impressions_90d  is_declining_label  
15470           0.0  0.0                1                   0  
15471          23.0  0.0                1                   0  
15482           9.0  0.0    

**WEAK PICKS:**

**Case 1:**
Some pages are very stale but receive very few impressions. Refreshing them may not produce enough benefit to justify the work.

**Case 2:**
Some pages have CTR only slightly below the 0.5% threshold. Their low CTR could simply be normal week-to-week variation rather than a true problem.

**Case 3:**
Some pages rank well but belong to evergreen topics. Even though they are old, updating them may not improve performance and could introduce unnecessary risk.


**LEAKAGE CHECK:**
I confirmed that is_declining_label (and the trend_direction it was derived from) is not used to calculate the score. We only used freshness_tier, avg_position, and ctr for the rule logic. This ensures our baseline isn't 'cheating' by looking at the future trend.

## Conclusion

This baseline uses two simple and explainable signals: content freshness and CTR performance. Both signals were checked before building the rule and showed evidence that they are useful indicators. The scoring system is easy to interpret and avoids using future information or label-derived features. Although this rule will not identify every declining page, it provides a strong baseline that can be compared with a machine learning model in the next stage of the project.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.